# Segmentación de clientes con K-Means

Dataset: Mall Customers. Las columnas quedan en inglés desde la carga: `customer_id`, `gender`, `age`, `annual_income_k`, `spending_score`.

El cuaderno muestra cada paso. La aplicación (FastAPI, Streamlit y la web) no repite esa lógica: llama a un solo orquestador a través de un solo controller.


## Parte 1 — Exploración

In [1]:
from mango.data.quality import load_customers, screen_customers, summarize_customers

raw = load_customers()
ready, quality = screen_customers(raw)
summary = summarize_customers(ready)
raw.head()


,customer_id,gender,age,annual_income_k,spending_score
0,1,Male,19,15,39
1,2,Male,21,15,81
2,3,Female,20,16,6
3,4,Female,23,16,77
4,5,Female,31,17,40


In [2]:
summary["numeric_summary"]


{'age': {'count': 200,
  'mean': 38.85,
  'std': 13.96900733155888,
  'min': 18.0,
  'q1': 28.75,
  'median': 36.0,
  'q3': 49.0,
  'max': 70.0},
 'annual_income_k': {'count': 200,
  'mean': 60.56,
  'std': 26.264721165271244,
  'min': 15.0,
  'q1': 41.5,
  'median': 61.5,
  'q3': 78.0,
  'max': 137.0},
 'spending_score': {'count': 200,
  'mean': 50.2,
  'std': 25.823521668370173,
  'min': 1.0,
  'q1': 34.75,
  'median': 50.0,
  'q3': 73.0,
  'max': 99.0}}

In [3]:
summary["gender_counts"]


{'Female': 112, 'Male': 88}

### ¿Qué variables son más adecuadas para segmentar clientes?

`annual_income_k` y `spending_score`. El ingreso mide capacidad de pago y el spending score mide la propensión a comprar. Juntas describen el valor del cliente para marketing.

No entran:

- `customer_id`: es un identificador. Una distancia entre códigos no significa nada.
- `gender`: es categórica. Meterla como número, o como dummy, deforma la distancia euclidiana de K-Means. El género se mira después, dentro de cada grupo.
- `age`: se describe por grupo, pero no define el segmento. Más abajo se compara su silueta y queda por debajo del par ingreso–gasto.


In [4]:
quality["iqr_fences"]


{'age': {'lower': -1.625, 'upper': 79.375, 'count': 0, 'customer_ids': []},
 'annual_income_k': {'lower': -13.25,
  'upper': 132.75,
  'count': 2,
  'customer_ids': [199, 200]},
 'spending_score': {'lower': -22.625,
  'upper': 130.375,
  'count': 0,
  'customer_ids': []}}

In [5]:
{
    "modified_z": quality["modified_z"],
    "mahalanobis_ids": quality["mahalanobis"]["customer_ids"],
    "rows_removed": quality["rows_removed"],
    "statistical_tail_count": quality["statistical_tail_count"],
}


{'modified_z': {'age': {'threshold': 3.5, 'count': 0, 'customer_ids': []},
  'annual_income_k': {'threshold': 3.5, 'count': 0, 'customer_ids': []},
  'spending_score': {'threshold': 3.5, 'count': 0, 'customer_ids': []}},
 'mahalanobis_ids': [3, 199, 200],
 'rows_removed': 0,
 'statistical_tail_count': 3}

### ¿Se identifican valores atípicos?

Sí, y no se borran.

No hay nulos, ni duplicados, ni valores imposibles (edad fuera de 15–100, ingreso negativo o spending score fuera de 1–100). Por eso `rows_removed` es 0.

La valla de Tukey (1.5 IQR) marca solo `annual_income_k` en `customer_id` 199 y 200. Los dos tienen ingreso 137, por encima del límite 132.75. La z modificada de Iglewicz y Hoaglin (umbral 3.5) no marca a nadie: la desviación absoluta mediana de estas columnas es ancha.

Mahalanobis, con una sola elipse y alfa 0.975, marca `customer_id` 3, 199 y 200. Esa prueba asume una nube gaussiana. Aquí la nube tiene varios grupos, así que la elipse señala esquinas de segmentos reales, no errores de captura.

La regla de industria es separar error de captura y cola real. 199 y 200 son clientes de ingreso alto. Quitarlos sería recortar el público que marketing quiere ver. Se etiquetan como `statistical_tail` y se quedan.


In [6]:
summary["correlations"]


{'age': {'age': 1.0,
  'annual_income_k': -0.012398,
  'spending_score': -0.327227},
 'annual_income_k': {'age': -0.012398,
  'annual_income_k': 1.0,
  'spending_score': 0.009903},
 'spending_score': {'age': -0.327227,
  'annual_income_k': 0.009903,
  'spending_score': 1.0}}

### ¿Qué tipo de relaciones se observan entre las variables?

No hay una recta entre ingreso y gasto: la correlación de Pearson es 0.010. La estructura es de grupos, no de pendiente. Por eso K-Means encaja mejor que una regresión.

`age` y `spending_score` tienen correlación −0.327: a mayor edad, el score tiende a bajar, pero la relación es débil. `age` y `annual_income_k` están prácticamente sueltas (−0.012).

Eso se ve en los cinco bloques del plano ingreso–gasto, no en un diagrama de dispersión con una sola nube.


## Parte 2 — Preprocesamiento

In [7]:
from mango.clustering.model import scale_features
from mango.constants import SEGMENT_FEATURES

scaled, scaler = scale_features(ready, SEGMENT_FEATURES)
list(zip(SEGMENT_FEATURES, scaler.mean_, scaler.scale_))


[('annual_income_k', np.float64(60.56), np.float64(26.19897707926781)),
 ('spending_score', np.float64(50.2), np.float64(25.7588819633151))]

### ¿Por qué es necesario escalar las variables en K-Means?

K-Means asigna cada punto al centroide más cercano con distancia euclidiana. Esa distancia suma unidades distintas como si fueran lo mismo. `annual_income_k` está en miles de dólares y `spending_score` es un índice de 1 a 100.

En este archivo las desviaciones son parecidas (cerca de 26.2 y 25.8), así que el daño práctico es pequeño. Aun así el algoritmo no es invariante a la escala, y las unidades no son las mismas. `StandardScaler` deja cada variable con media 0 y desviación 1.

Si más adelante entrara `age` (desviación cerca de 14), sin escalar pesaría menos que el ingreso. Se elige primero el par de negocio y después se escala ese par.


## Parte 3 — Método del codo

In [8]:
from mango.clustering.model import elbow_curve, geometric_elbow_k
from mango.visualization import plot_elbow

elbow = elbow_curve(scaled)
print("elbow_k", geometric_elbow_k(elbow))
for point in elbow:
    print(point["k"], round(point["inertia"], 1))


elbow_k 5
1 400.0
2 269.7
3 157.7
4 108.9
5 65.6
6 55.1
7 44.9
8 37.2
9 32.4
10 30.0


### ¿Por qué valores cercanos a cero indican mejor ajuste?

La inercia es la suma de distancias al cuadrado de cada punto a su centroide (WCSS). Con las dos variables ya estandarizadas, K = 1 vale 400: 200 clientes por 2 variables de varianza 1. K = 5 baja a 65.6 y K = 10 sigue en 30.0.

Cerca de cero, los puntos están pegados a su centro. Cero exacto aparecería si cada cliente fuera su propio cluster. Eso ajusta la muestra y no deja un segmento usable. La inercia sola siempre premia un K más grande.


### ¿Cómo se identifica el punto óptimo de K?

Se busca el codo: el K donde la curva deja de caer fuerte. Además del ojo, se mide el punto más lejos de la cuerda que une el primer y el último valor. Ese codo geométrico cae en K = 5. De 4 a 5 la inercia pasa de 108.9 a 65.6; de 5 a 6 solo baja a 55.1. La ganancia se aplana.


## Parte 4 — Coeficiente de silueta

In [9]:
from mango.clustering.model import silhouette_curve

silhouette = silhouette_curve(scaled)
for point in silhouette:
    print(point["k"], round(point["score"], 4))


2 0.3213
3 0.4666
4 0.4939
5 0.5547
6 0.5399
7 0.5281
8 0.4552
9 0.4571
10 0.4432


### ¿Qué mide el coeficiente de silueta?

Para cada punto compara dos distancias. `a` es la distancia media a los demás puntos de su cluster. `b` es la distancia media al cluster vecino más cercano. El coeficiente es `(b - a) / max(a, b)`. El número del modelo es el promedio de todos los puntos.


### ¿Qué significa un valor cercano a 1?

El punto está cerca de su grupo y lejos de los demás. Los clusters son compactos y están separados. En este modelo el máximo es 0.555 en K = 5: una estructura clara, no un separación perfecta.


### ¿Qué implica un valor cercano a 0 o negativo?

Cerca de 0, el punto está en la frontera entre dos grupos. Negativo, está más cerca de otro cluster que del suyo: el K o las variables están mal elegidos. Aquí el peor valor del rango es K = 2, con 0.321. Ninguno es negativo.


### ¿Coincide el valor óptimo de K con el método del codo?

Sí. El codo geométrico y el máximo de silueta caen en K = 5. La silueta de K = 6 es 0.540, por debajo de 0.555. La regla guardada es `elbow_and_silhouette_agree`.

El mismo K gana al comparar conjuntos de variables. Ingreso y spending score en K = 5 dan 0.555. Edad y gasto dan 0.472, edad e ingreso 0.443, y las tres juntas 0.428. Por eso el modelo publicado usa solo `annual_income_k` y `spending_score`.


## Parte 5 — Modelo K-Means

In [10]:
from mango.orchestration.pipeline import SegmentationOrchestrator

orchestrator = SegmentationOrchestrator()
analysis = orchestrator.fit()
analysis["selected_k"], round(analysis["model_silhouette"], 4), analysis["k_selection"]["rule"]


(5, 0.5547, 'elbow_and_silhouette_agree')

In [11]:
[
    {
        "profile": cluster["profile"],
        "size": cluster["size"],
        "annual_income_k": round(cluster["centroid"]["annual_income_k"], 1),
        "spending_score": round(cluster["centroid"]["spending_score"], 1),
        "mean_age": round(cluster["mean_age"], 1),
    }
    for cluster in analysis["clusters"]
]


[{'profile': 'high_value',
  'size': 39,
  'annual_income_k': 86.5,
  'spending_score': 82.1,
  'mean_age': 32.7},
 {'profile': 'potential',
  'size': 35,
  'annual_income_k': 88.2,
  'spending_score': 17.1,
  'mean_age': 41.1},
 {'profile': 'average',
  'size': 81,
  'annual_income_k': 55.3,
  'spending_score': 49.5,
  'mean_age': 42.7},
 {'profile': 'budget_constrained',
  'size': 22,
  'annual_income_k': 25.7,
  'spending_score': 79.4,
  'mean_age': 25.3},
 {'profile': 'low_value',
  'size': 23,
  'annual_income_k': 26.3,
  'spending_score': 20.9,
  'mean_age': 45.2}]

### ¿Qué representa cada cluster?

El nombre sale del cuadrante del centroide respecto de la mediana de la muestra. El centroide más cercano a esa mediana es el grupo promedio.

| profile | n | annual_income_k | spending_score | mean_age |
|---|---:|---:|---:|---:|
| high_value | 39 | 86.5 | 82.1 | 32.7 |
| potential | 35 | 88.2 | 17.1 | 41.1 |
| average | 81 | 55.3 | 49.5 | 42.7 |
| budget_constrained | 22 | 25.7 | 79.4 | 25.3 |
| low_value | 23 | 26.3 | 20.9 | 45.2 |

`high_value` es ingreso alto y gasto alto. `potential` es ingreso alto y gasto bajo. `average` es el centro. `budget_constrained` gasta mucho con ingreso bajo. `low_value` está bajo en las dos.


### ¿Qué diferencias existen entre los grupos obtenidos?

El ingreso separa dos mundos: cerca de 26 mil dólares al año contra cerca de 87. Dentro de cada mundo, el spending score separa a quien compra de quien no. El grupo promedio es el más grande (81 personas) y se queda en el medio de las dos variables.

La edad no define el cluster, pero describe. `budget_constrained` es el más joven (media 25.3). `low_value` y `average` pasan de 42. La mezcla de género queda entre 46 % y 61 % de `Female`: ningún segmento es de un solo género, así que no se diseña la campaña como si lo fuera.

Quitar las dos colas de ingreso (199 y 200) mueve la silueta de 0.555 a 0.560 y el centroide, en unidades originales, como máximo 1.9. No cambian el mapa. Se quedan en el modelo.


## Parte 6 — Interpretación de negocio

In [12]:
for cluster in analysis["clusters"]:
    print(cluster["profile"])
    print(cluster["strategy"])
    print()
print("growth_opportunity", analysis["growth_opportunity"])


high_value
Fidelización, acceso anticipado y servicio personal. Evitar descuentos agresivos que erosionen el margen.

potential
Campañas de conversión: experiencias, personal shopper y surtido aspiracional. Una visita nueva en este grupo mueve más el ingreso que en el resto.

average
Retención con campañas de temporada y venta cruzada moderada. Medir la respuesta antes de subir la inversión.

budget_constrained
Promociones, financiación y formatos accesibles. El volumen no se convierte solo en margen: cuidar el descuento.

low_value
Comunicación de precio y surtido básico. No concentrar el presupuesto de adquisición en este grupo.

growth_opportunity potential


### ¿Qué estrategias de marketing se podrían aplicar a cada grupo?

- **high_value.** Fidelización, acceso anticipado y servicio personal. No quemar margen con descuentos.
- **potential.** Conversión: experiencias, personal shopper y surtido aspiracional. Ya tienen capacidad de pago y casi no consumen en el centro comercial.
- **average.** Campañas de temporada y venta cruzada corta. Medir antes de subir la inversión.
- **budget_constrained.** Precio, financiación y formatos accesibles. Compran seguido, pero el ticket tiene techo.
- **low_value.** Comunicación de precio y surtido básico. No poner ahí el presupuesto de adquisición.


### ¿Qué segmento representa mayor oportunidad de crecimiento?

`potential`: 35 clientes, centroide de ingreso 88.2 y spending score 17.1. El centro comercial ya los tiene identificados y su ingreso es el más alto del mapa, pero el gasto es de los más bajos. Cada visita nueva mueve más ingreso que empujar a `budget_constrained`, que ya gasta mucho y no tiene de dónde subir el ticket.

`high_value` se defiende; no es crecimiento, es retención.


## Parte 7 — Despliegue

La app de Streamlit y la web usan el mismo `SegmentationController`. Streamlit muestra el codo, los clusters y clasifica un cliente nuevo. La web hace lo mismo contra la API, y si la API no está usa el `analysis.json` exportado.

```bash
uv run streamlit run streamlit_app.py
uv run fastapi dev
```
